# Base versus tuned policy comparison

Saved outputs in this source match the thesis spiral and square tracking figures byte for byte. Execution/hardware cells are reference text only or omitted.

Selected source excerpts from the original research notebook. Outputs, execution counts, widget state and duplicate plotting trials are removed. Definitions are kept as code; configuration, training launches and hardware calls are displayed as reference text so Run All cannot start training or connect to a robot. This is a review extract, not a complete reproducibility package. See README.md for omitted dependencies.


In [ ]:
import os
import random
from collections import deque
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
device = torch.device("cpu")


## Source cell 3 (original zero-based index 2)


Reference only; not executed.

```python
import os
import torch
import random
import numpy as np
import torch.nn as nn
from tqdm import tqdm
import torch.optim as optim
from scipy.io import loadmat
from collections import deque

from utils import RobotCMD
from utils import Cameras
from utils import ExperimentalEnv

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd

# --- Professional Plotting Style Settings ---
plt.rcParams.update({
    'font.size': 12, 'axes.titlesize': 14, 'axes.labelsize': 12,
    'xtick.labelsize': 10, 'ytick.labelsize': 10, 'legend.fontsize': 10,
    'lines.linewidth': 2.0, 'font.family': 'sans-serif', 'axes.grid': True,
    'grid.alpha': 0.5, 'grid.linestyle': '--', 'figure.autolayout': True 
})

# Configuration
current_dir = os.getcwd()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

# Path configurations
EXP_NAME = '500 eps-Rand-cube-10 HZ'
PATHS = {
    # Weights (Base/Pre-trained and Fine-Tuned)
    'best_weights': os.path.join(current_dir, 'Weights', EXP_NAME, 'Best_weights'),
    'last_weights': os.path.join(current_dir, 'Weights', EXP_NAME, 'Last_weights'),
    'fine_tuned_weights': os.path.join(current_dir, 'Weights', EXP_NAME, 'FineTuned_weights'),

    # Renders and Plots
    'render_tests': os.path.join(current_dir, 'Renders', EXP_NAME, 'Test_DRL'),
    'plots_test': os.path.join(current_dir, 'Plots', EXP_NAME, 'Tests_Disturbances'),

    # Data saving target directory
    'variables_test': os.path.join(current_dir, 'Variables', EXP_NAME, 'Test'),

    # Hardware Specific & MATLAB
    'camera': os.path.join(current_dir, 'Cameras'),
    'matlab_workspaces': os.path.abspath(os.path.join(current_dir, '2) Workspaces', 'Cartesian'))
}

ROBOT_IP = "ROBOT_HOST_FROM_LOCAL_CONFIGURATION"
ROBOT_PORT = 8055
CONT_DT = 0.1  # 10 Hz

# Create all necessary directories
for path in PATHS.values():
    os.makedirs(path, exist_ok=True)
```


## Source cell 9 (original zero-based index 8)


In [ ]:
class Actor(nn.Module):
    def __init__(self, actor_input_dim, action_dim, action_max, action_min, actor_hidden, input_min, input_max):
        super().__init__()
        self.register_buffer('action_max', torch.tensor(action_max, dtype=torch.float32))
        self.register_buffer('action_min', torch.tensor(action_min, dtype=torch.float32))
        self.register_buffer('input_min', torch.tensor(input_min, dtype=torch.float32))
        self.register_buffer('input_max', torch.tensor(input_max, dtype=torch.float32))
        layers = []
        for size in actor_hidden:
            layers.extend([nn.Linear(actor_input_dim, size), nn.LayerNorm(size), nn.ReLU()])
            actor_input_dim = size
        layers.append(nn.Linear(actor_input_dim, action_dim))
        self.net = nn.Sequential(*layers).to(device)
        self.apply(self._init_weights)
   
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
            nn.init.zeros_(m.bias)
   
    def forward(self, x):
        x_norm = 2 * (x - self.input_min) / (self.input_max - self.input_min) - 1
        act_norm = torch.tanh(self.net(x_norm))
        act = 0.5 * (act_norm + 1) * (self.action_max - self.action_min) + self.action_min
        return act


## Source cell 15 (original zero-based index 14)


Reference only; not executed.

```python
# Initialize Environment Parameters
state_dim, action_dim = env.observation_space.shape[0], env.action_space.shape[0]           
act_low, act_high = env.action_space.low, env.action_space.high
state_low, state_high = env.observation_space.low, env.observation_space.high

# --- Initialize BASE (Not Tuned) Actor ---
actor_base = Actor(state_dim, action_dim, act_high, act_low, (400, 300), state_low, state_high).to(device)
weights_path_base = os.path.join(PATHS['best_weights'], 'actor_dict.pth')
actor_base.load_state_dict(torch.load(weights_path_base, map_location=device))
actor_base.eval() 

# --- Initialize FINE-TUNED Actor ---
actor_tuned = Actor(state_dim, action_dim, act_high, act_low, (400, 300), state_low, state_high).to(device)
weights_path_tuned = os.path.join(PATHS['fine_tuned_weights'], 'actor_dict_best.pth')
actor_tuned.load_state_dict(torch.load(weights_path_tuned, map_location=device))
actor_tuned.eval()

print("Environment initialized. Both Base and Fine-Tuned DRL Actor weights loaded successfully.")
```


## Source cell 18 (original zero-based index 17)


In [ ]:
class DRLSimulator:
    """
    Manages the physical simulation of a DRL actor tracking target trajectories.
    Refactored to remove downsampling and focus directly on dataset tracking.
    """
    def __init__(self, env, actor, device, seed=42):
        self.env = env
        self.actor = actor
        self.device = device
        self.seed = seed
        self.act_low = env.action_space.low
        self.act_high = env.action_space.high

    def _prepare_targets(self, file_path):
        """Loads physical trajectory from a dataset mat file."""
        datas = loadmat(file_path)
        targets = datas['datas']['Pos'][0, 0]
        return targets

    def run_simulation(self, file_path):
        """
        Executes hardware tracking across the provided target file.
        Returns:
            (actual_positions, target_positions) as numpy arrays.
        """
        np.random.seed(self.seed)
        targets = self._prepare_targets(file_path)
        
        # Hardware Environment Reset
        _ = self.env.reset(action0=np.array([0.0, 0.0, 0.20]), render=True)
        
        actual_positions = []
        target_positions = []

        print(f"Tracking trajectory: {len(targets)} steps...")
        for step, target in enumerate(tqdm(targets, desc="Simulation Steps")):
            # 1. Provide setpoint to Environment
            self.env.p_target = target
            self.env.camera.update_setpoint(env.p_target)
            self.env.error_tip_t = self.env.p_tip_t - target
            state = self.env._get_state()

            # 2. Query Agent for Action
            with torch.no_grad():
                state_t = torch.as_tensor(state, dtype=torch.float32, device=self.device)
                current_raw_action = self.actor(state_t).cpu().numpy()

            action_to_apply = np.clip(current_raw_action, self.act_low, self.act_high)

            # 3. Apply to Hardware & Log
            self.env.step(action_to_apply)
            
            actual_positions.append(self.env.p_tip_t.copy())
            target_positions.append(target.copy())

        return np.array(actual_positions), np.array(target_positions)

def display_metrics(metrics_dict, title=""):
    """Displays a dictionary of metrics in a formatted dataframe."""
    if title:
        print(f"\n--- {title} ---")
    df = pd.Series(metrics_dict).to_frame(name='Value').round(4)
    print(df)
    print("-" * (len(title) + 6) if title else "---")


## Source cell 22 (original zero-based index 21)


Reference only; not executed.

```python
# --- Load Data & Calculate Metrics ---

def calculate_xy_metrics(actual_traj, target_traj):
    """Calculates RMSE for X and Y axes and their Mean Error ($RMSE = \sqrt{\frac{1}{N}\sum (Actual-Target)^2}$)."""
    if actual_traj.shape != target_traj.shape:
        raise ValueError("Shape mismatch between actual and target trajectories.")
        
    error_mm = (actual_traj - target_traj) * 1000  # Convert directly to millimeters
    
    rmse_x = np.sqrt(np.mean(error_mm[:, 0] ** 2))
    rmse_y = np.sqrt(np.mean(error_mm[:, 1] ** 2))
    mean_rmse_xy = np.mean([rmse_x, rmse_y])
    
    return {
        "RMSE_X (mm)": rmse_x,
        "RMSE_Y (mm)": rmse_y,
        "Mean_XY_RMSE (mm)": mean_rmse_xy
    }

save_path_base = os.path.join(PATHS['variables_test'], 'Dynamic_PDMS2_Spiral_Set3_0_base_data.npz')
save_path_tuned = os.path.join(PATHS['variables_test'], 'Dynamic_PDMS2_Spiral_Set3_0_tuned_data.npz')

# 1. Load the exported arrays
data_base = np.load(save_path_base)
data_tuned = np.load(save_path_tuned)

act_base = data_base['actual']
act_tuned = data_tuned['actual']
target_pos = data_base['target'] # Target is the same for both

# 2. Generate and display metrics
metrics_base = calculate_xy_metrics(act_base, target_pos)
metrics_tuned = calculate_xy_metrics(act_tuned, target_pos)

display_metrics(metrics_base, title="BASE MODEL (Not Tuned) Metrics")
display_metrics(metrics_tuned, title="FINE-TUNED MODEL Metrics")
```


## Source cell 24 (original zero-based index 23)


Reference only; not executed.

```python
import os
import numpy as np
import matplotlib.pyplot as plt

def plot_model_comparison(actual_base, actual_tuned, target, sim_dt=0.01, save_dir=None):
    # Setup for 2 models (Base and Tuned) equivalent to num_vels = 2
    num_models = 2
    fig = plt.figure(figsize=(10, 4 * num_models))
    
    # GridSpec exactly as in the reference
    gs = fig.add_gridspec(
        2 * num_models, 2, height_ratios=[1] * (2 * num_models), 
        width_ratios=[1, 1], hspace=0.2, wspace=0.01
    )
    
    ax_trajs, ax_time_xs, ax_time_ys = [], [], []
    for i in range(num_models):
        ax_trajs.append(fig.add_subplot(gs[2*i : 2*i+2, 0]))
        ax_time_xs.append(fig.add_subplot(gs[2*i, 1]))
        ax_time_ys.append(fig.add_subplot(gs[2*i+1, 1]))
        
    handles, labels = [], []
    
    # Package data to loop through exactly like the velocities loop
    models_data = [
        {'name': 'Base Model', 'actual': actual_base, 'color': 'b', 'label': 'Base Agent'},
        {'name': 'Fine-Tuned Model', 'actual': actual_tuned, 'color': 'g', 'label': 'Tuned Agent'}
    ]
    
    time_steps = np.arange(len(target)) * sim_dt
    
    for idx, model in enumerate(models_data):
        ax_traj = ax_trajs[idx]
        actual_drl = model['actual']
        m_color = model['color']
        m_label = model['label']
        
        mark_every = max(1, len(actual_drl) // 10)
        
        # --- Trajectory Plot (Left Column) ---
        line_target, = ax_traj.plot(
            target[:, 0] * 100, target[:, 1] * 100, 
            'r-', label='Desired', lw=2.5, markersize=10, markevery=mark_every, marker='.', alpha=1.0
        )
        line_drl, = ax_traj.plot(
            actual_drl[:, 0] * 100, actual_drl[:, 1] * 100, 
            f'{m_color}--', label=m_label, lw=2.5, markersize=6, markevery=mark_every, marker='s', alpha=0.8
        )
        
        # End points scatter
        ax_traj.scatter(
            actual_drl[-1, 0] * 100, actual_drl[-1, 1] * 100, 
            c=m_color, marker='s', s=80, edgecolors=m_color, lw=1.5, zorder=5
        )
        ax_traj.scatter(
            target[-1, 0] * 100, target[-1, 1] * 100, 
            c='r', marker='.', s=100, edgecolors='r', lw=1.5, zorder=5
        )
        
        # Setup legend handles from the first plot
        if idx == 0: 
            # We capture Base and Desired handles initially, but will update handles below for the global legend
            handles, labels = [line_drl, line_target], [m_label, 'Desired']
        if idx == 1:
            # Add the Tuned agent handle to the global legend list
            handles = [handles[0], line_drl, handles[1]] 
            labels = [labels[0], m_label, labels[1]]
            
        if idx == num_models - 1: 
            ax_traj.set_xlabel('X Position (cm)', fontsize=10)
            
        ax_traj.set_ylabel('Y Position (cm)', fontsize=10)
        ax_traj.set_aspect('equal', adjustable='box')
        ax_traj.grid(True, alpha=0.3)
        
        # Text box for Model Name
        ax_traj.text(
            0.98, 0.98, f"{model['name']}", transform=ax_traj.transAxes, ha='right', va='top', 
            fontsize=10, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8), zorder=10
        )
        
        # --- Time Series X (Right Column, Top) ---
        ax_time_xs[idx].plot(time_steps, target[:, 0] * 100, '-r', lw=1.5, marker='.', markevery=mark_every)
        ax_time_xs[idx].plot(time_steps, actual_drl[:, 0] * 100, f'--{m_color}', lw=1.5, marker='s', markevery=mark_every)
        ax_time_xs[idx].set_ylabel('X (cm)', fontsize=10)
        ax_time_xs[idx].yaxis.tick_right()
        ax_time_xs[idx].yaxis.set_label_position("right")
        ax_time_xs[idx].grid(True, alpha=0.3)
        
        # --- Time Series Y (Right Column, Bottom) ---
        ax_time_ys[idx].plot(time_steps, target[:, 1] * 100, '-r', lw=1.5, marker='.', markevery=mark_every)
        ax_time_ys[idx].plot(time_steps, actual_drl[:, 1] * 100, f'--{m_color}', lw=1.5, marker='s', markevery=mark_every)
        ax_time_ys[idx].set_ylabel('Y (cm)', fontsize=10)
        ax_time_ys[idx].yaxis.tick_right()
        ax_time_ys[idx].yaxis.set_label_position("right")
        ax_time_ys[idx].grid(True, alpha=0.3)
        
        if idx == num_models - 1: 
            ax_time_ys[idx].set_xlabel('Time (s)', fontsize=10)

    # --- Final Polish ---
    fig.suptitle('Performance Comparison: Base vs. Fine-Tuned Model', fontsize=16, y=1.0)
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.98), ncol=3, fontsize=10)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    
    if save_dir:
        plt.savefig(os.path.join(save_dir, 'Spiral_Model_Comparison_Grid.png'), dpi=300, bbox_inches='tight')
        
    plt.show()

# Generate the plot
plot_model_comparison(act_base, act_tuned, target_pos, sim_dt=CONT_DT, save_dir=PATHS['plots_test'])
```
